# Ablation Study — Hybrid CNN-Transformer Crowd Counting on ShanghaiTech

This notebook measures how much each component of the hybrid model contributes to its accuracy
by training variants with that component removed and comparing them to the full model
**under an identical training budget**.

| Variant | What is removed / changed |
|---|---|
| `full` | Nothing: VGG16 + multi-scale fusion + DPPM + STN + Transformer + graph occlusion module |
| `no_multiscale` | Multi-scale fusion (only `conv4_3` features feed the head, `conv3_3` branch dropped) |
| `no_dppm` | Dilated Pyramid Pooling Module |
| `no_stn` | Spatial Transformer Network |
| `no_transformer` | Transformer encoder (global self-attention) |
| `no_graph` | Graph-based occlusion reasoning module |
| `baseline` | All five modules removed: VGG16 + density decoder only |
| `inc_*` *(optional)* | Cumulative study: baseline, then modules added one at a time |
| `loss_*` *(optional)* | Loss ablations: MSE only, without SSIM, without count loss |

Everything else is copied **unchanged** from the main notebook (data split with seed 42,
geometry-adaptive density maps, augmentation, loss, optimiser, LR schedule, overlapping tiled
inference, flip TTA at test time), so the only difference between rows of the final table is
the component being ablated.

### How to use it
1. `Runtime → Change runtime type → GPU` (T4 is fine).
2. Run the cells top to bottom. Google Drive is mounted so **every run checkpoints to Drive
   after every epoch**. If Colab disconnects, just reconnect and *Run all* again: finished
   variants are skipped and an interrupted variant resumes from its last epoch.
3. When all variants are done, Section 9 builds the results table (CSV + LaTeX), the bar chart,
   paired significance tests, and a qualitative figure for the manuscript.

### Time budget — read before starting
One 200-epoch run of the full model took several hours on a T4. Seven variants at 200 epochs
would not fit in Colab sessions, so the default here is **`ABLATION_EPOCHS = 100`** (the main
run reached its best validation MAE at epoch 68). The notebook prints the time per epoch and
an ETA after the first epoch of each variant; lower `ABLATION_EPOCHS` or trim `RUN_VARIANTS`
in Section 2 if needed.

> **Important for the paper:** compare ablation rows only with the `full` row *from this
> notebook* (same epochs, same seed). Do not mix them with the 200-epoch result from the main
> notebook (test MAE 79.07); report that one separately as the final model result.

## 1. Environment setup

In [ ]:
!pip -q install kagglehub h5py --upgrade

In [ ]:
import os
import math
import glob
import json
import random
import zipfile
import shutil
from dataclasses import dataclass, asdict

import numpy as np
import cv2
from PIL import Image
import scipy.io as sio
from scipy.spatial import KDTree

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import torchvision.transforms as T
import torchvision.models as tv_models

import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# bfloat16 has the same exponent range as float32, so activations cannot overflow the way they
# do in float16 (max ~65504 -> inf -> NaN through BatchNorm). Prefer it whenever the GPU supports
# it, including T4, where it is emulated: a bit slower, but a float16 run on T4 produced NaN
# losses from the very first step. float16 + GradScaler is only the fallback for GPUs without
# bf16 support. The custom layers after VGG run in float32 regardless (see the model's forward).
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("Mixed-precision dtype:", AMP_DTYPE)


import time
import csv
import pandas as pd

In [ ]:
# Results and checkpoints go to Google Drive so they survive Colab disconnects.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = "/content/drive/MyDrive/crowd_counting_ablation"
else:
    RUN_ROOT = "/content/crowd_counting_ablation"   # lost when the runtime is recycled
os.makedirs(RUN_ROOT, exist_ok=True)
print("Ablation outputs ->", RUN_ROOT)

## 2. Configuration

The first cell is the main notebook's `CFG`, unchanged. The second cell holds the
ablation-specific settings: which variants to run, how many epochs, and which seeds.

In [ ]:
@dataclass
class CFG:
    # ---- dataset ----
    part: str = "A"                      # "A", "B", or "both" (combine both parts for
                                          # more training data -- see the caveat printed
                                          # in Section 3 about their differing domains)
    dataset_search_roots: tuple = (
        "/content/ShanghaiTech", "/content/shanghaitech",
        "/content/drive/MyDrive/ShanghaiTech",
    )
    density_cache_root: str = "/content/density_cache"
    val_split: float = 0.15              # held out from the official *training* split; kept
                                          # larger than a typical 90/10 split because ShanghaiTech
                                          # per-image counts vary hugely (~100s to ~1500+), so a
                                          # small val set makes the epoch-to-epoch MAE too noisy
                                          # for the scheduler/checkpointing to read reliably

    # ---- image / patch handling ----
    patch_size: int = 256                # multiple of output_stride; used for train crops AND tiled inference
    crops_per_image: int = 4             # [PERF FIX] random crops drawn per training image per epoch.
                                          # With 1 crop/image an epoch was only ~31 steps and saw ~1/8
                                          # of each Part A image -> the model under-fitted badly
                                          # (train patch-MAE ~40 on patches averaging ~60 people)
    scale_aug: tuple = (0.7, 1.3)        # [PERF FIX] random, count-preserving rescale of each training
                                          # image -- head sizes in ShanghaiTech vary enormously and
                                          # flip-only augmentation left the model scale-brittle
    gray_aug_prob: float = 0.1           # [PERF FIX] Part A contains some grayscale images
    infer_overlap: float = 0.5           # [PERF FIX] fraction of overlap between sliding-window tiles
    output_stride: int = 8               # backbone downsampling factor (stride of the density head)
    density_scale: float = 100.0         # scales tiny per-pixel density values for stable gradients

    # ---- model ----
    pretrained_backbone: bool = True
    freeze_backbone_stage1: bool = True  # freeze first conv block of VGG16 (memory + fewer params to update)
    transformer_layers: int = 2
    transformer_heads: int = 4
    transformer_ff_dim: int = 512
    graph_grid_size: int = 16
    graph_k: int = 8
    use_grad_checkpoint: bool = False    # flip on if you hit OOM on deep backbones
    use_torch_compile: bool = True       # torch.compile() the model for faster train/inference

    # ---- optimization ----
    batch_size: int = 8
    num_workers: int = 2
    base_lr: float = 1e-4                # LR of the newly-initialised modules (DPPM/STN/transformer/...)
    backbone_lr_mult: float = 0.1        # [PERF FIX] pretrained VGG layers use base_lr * 0.1 so their
                                          # ImageNet features aren't wrecked by the new heads' large,
                                          # noisy early gradients
    weight_decay: float = 1e-4
    epochs: int = 200                    # each epoch is now crops_per_image (4x) longer, so 200
                                          # epochs = ~3.2x more optimisation steps than the old 250
    grad_accum_steps: int = 1
    amp: bool = True                     # automatic mixed precision
    grad_clip_norm: float = 5.0          # caps gradient norm to prevent exploding-gradient NaNs

    # ---- loss weights ----
    lambda_mse: float = 1.0
    lambda_ssim: float = 0.1
    lambda_count: float = 0.02

    # ---- scheduler ----
    lr_factor: float = 0.5
    lr_patience: int = 10                # epochs (of the *smoothed* val MAE) without improvement
    val_mae_smoothing: int = 5           # [PERF FIX] ReduceLROnPlateau is driven by a moving average of
                                          # the last N val MAEs. Driving it by the raw, very noisy
                                          # 45-image val MAE decayed the LR to its 1e-6 floor long
                                          # before the model had converged
    lr_min: float = 1e-6
    warmup_epochs: int = 3                # linear LR ramp-up before ReduceLROnPlateau engages,
                                          # so the scheduler's "best" tracking starts once
                                          # training has actually stabilized

    # ---- inference ----
    use_tta: bool = True                 # average a prediction with its horizontal-flip
                                          # counterpart at test/inference time (not during
                                          # per-epoch validation, to keep training fast)

    # ---- checkpoints ----
    ckpt_dir: str = "/content/checkpoints"


cfg = CFG()
os.makedirs(cfg.ckpt_dir, exist_ok=True)
os.makedirs(cfg.density_cache_root, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2, default=str))

In [ ]:
# ---------------- ablation settings ----------------
ABLATION_EPOCHS = 100          # same budget for every variant (see the time-budget note above)
SEEDS = [42]                   # e.g. [42, 43, 44] to report mean ± std (3x the compute)

# Leave-one-out study (the core table reviewers expect). "full" runs first so the reference
# row always exists even if you stop part-way.
RUN_VARIANTS = ["full", "baseline", "no_transformer", "no_graph", "no_dppm", "no_stn",
                "no_multiscale"]

# Optional extra studies -- uncomment to append them to the queue.
# RUN_VARIANTS += ["inc_ms", "inc_ms_dppm", "inc_ms_dppm_stn", "inc_ms_dppm_stn_tr"]  # cumulative
# RUN_VARIANTS += ["loss_mse_only", "loss_no_ssim", "loss_no_count"]                 # loss terms

KEEP_LAST_CHECKPOINT = False   # delete each run's resume checkpoint once it finishes (saves Drive space)
PLOT_EVERY = 5                 # redraw the training curves every N epochs

cfg.epochs = ABLATION_EPOCHS
cfg.ckpt_dir = RUN_ROOT
print(f"{len(RUN_VARIANTS)} variants x {len(SEEDS)} seed(s) x {cfg.epochs} epochs")

## 3. Get the ShanghaiTech dataset

Same Kaggle route as the main notebook (`tthien/shanghaitech`). If you use Drive or a manual
upload instead, set `DATASET_PATH` to the extracted folder.

In [ ]:
# --- Option A: Kaggle (recommended) ------------------------------------------------
# Upload your kaggle.json first:
# from google.colab import files
# files.upload()  # select kaggle.json
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

DATASET_PATH = None
try:
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download("tthien/shanghaitech")
    print("Downloaded to:", DATASET_PATH)
except Exception as e:
    print("Kaggle download unavailable/failed:", e)
    print("Use Option B (Google Drive) or Option C (manual upload) below instead.")

In [ ]:
def find_dataset_root(search_roots):
    candidates = [r for r in search_roots if r and os.path.isdir(r)]
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if "part_A" in dirnames or "part_B" in dirnames:
                return dirpath
    # fall back: look for any directory that itself is named part_A/part_B
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath) in ("part_A", "part_B"):
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not locate part_A / part_B under: " + ", ".join(candidates)
        + ". Run one of the download options above first."
    )


def resolve_part_dirs(dataset_root, part):
    part_dir = os.path.join(dataset_root, f"part_{part}")
    train_img_dir = os.path.join(part_dir, "train_data", "images")
    train_gt_dir = os.path.join(part_dir, "train_data", "ground-truth")
    test_img_dir = os.path.join(part_dir, "test_data", "images")
    test_gt_dir = os.path.join(part_dir, "test_data", "ground-truth")
    for d in (train_img_dir, train_gt_dir, test_img_dir, test_gt_dir):
        assert os.path.isdir(d), f"Missing expected directory: {d}"
    return train_img_dir, train_gt_dir, test_img_dir, test_gt_dir


search_roots = list(cfg.dataset_search_roots)
if DATASET_PATH:
    search_roots = [DATASET_PATH] + search_roots

DATASET_ROOT = find_dataset_root(search_roots)
print("Dataset root:", DATASET_ROOT)

active_parts = ["A", "B"] if cfg.part == "both" else [cfg.part]

# Per-part image lists and ground-truth dirs -- kept separate (rather than merged into one
# flat list) because Part A uses geometry-adaptive Gaussians and Part B uses a fixed sigma
# (see precompute_density_maps in Section 4), and each part's images live in their own
# density-cache directory.
train_image_paths_by_part, test_image_paths_by_part = {}, {}
train_gt_dir_by_part, test_gt_dir_by_part = {}, {}
for p in active_parts:
    train_img_dir, train_gt_dir, test_img_dir, test_gt_dir = resolve_part_dirs(DATASET_ROOT, p)
    train_image_paths_by_part[p] = sorted(glob.glob(os.path.join(train_img_dir, "*.jpg")))
    test_image_paths_by_part[p] = sorted(glob.glob(os.path.join(test_img_dir, "*.jpg")))
    train_gt_dir_by_part[p] = train_gt_dir
    test_gt_dir_by_part[p] = test_gt_dir
    print(f"part_{p}: {len(train_image_paths_by_part[p])} train images, "
          f"{len(test_image_paths_by_part[p])} test images")

if cfg.part == "both":
    print(
        "\nNote: Part A (dense, web-crawled, avg ~501 people/image, variable resolution) and "
        "Part B (sparser, fixed surveillance camera, avg ~123 people/image) are different "
        "domains -- the literature always reports them as separate benchmarks, never pooled. "
        "Combining them gives the model roughly 2x the images and more viewpoint diversity, at "
        "the cost of a more heterogeneous training signal. Section 11 reports test MAE/RMSE "
        "per part as well as combined, so you can check whether either part is hurt by mixing."
    )

## 4. Ground-truth density maps (unchanged)

In [ ]:
def load_gt_points(mat_path):
    mat = sio.loadmat(mat_path)
    points = mat["image_info"][0, 0][0, 0][0]
    return points.astype(np.float32)


def _gaussian_kernel_2d(sigma):
    radius = max(1, int(3 * sigma))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum() + 1e-12
    return kernel, radius


def generate_density_map(img_hw, points, adaptive=True, k=3, beta=0.3,
                          fixed_sigma=15.0, max_sigma=20.0):
    h, w = img_hw
    density = np.zeros((h, w), dtype=np.float32)
    n = points.shape[0]
    if n == 0:
        return density

    if adaptive and n > k:
        tree = KDTree(points)
        dists, _ = tree.query(points, k=k + 1)
        sigmas = dists[:, 1:].mean(axis=1) * beta
        sigmas = np.clip(sigmas, 1.0, max_sigma)
    else:
        sigmas = np.full(n, fixed_sigma, dtype=np.float32)

    for i in range(n):
        x = int(round(float(points[i, 0])))
        y = int(round(float(points[i, 1])))
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        kernel, radius = _gaussian_kernel_2d(sigmas[i])
        x0, x1 = x - radius, x + radius + 1
        y0, y1 = y - radius, y + radius + 1
        kx0, kx1 = max(0, -x0), kernel.shape[1] - max(0, x1 - w)
        ky0, ky1 = max(0, -y0), kernel.shape[0] - max(0, y1 - h)
        cx0, cx1 = max(0, x0), min(w, x1)
        cy0, cy1 = max(0, y0), min(h, y1)
        density[cy0:cy1, cx0:cx1] += kernel[ky0:ky1, kx0:kx1]
    return density


def precompute_density_maps(image_paths, gt_dir, out_dir, part="A"):
    os.makedirs(out_dir, exist_ok=True)
    adaptive = (part == "A")
    for img_path in tqdm(image_paths, desc=f"Density maps -> {out_dir}"):
        fname = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(out_dir, fname + ".npy")
        if os.path.exists(out_path):
            continue
        gt_path = os.path.join(gt_dir, "GT_" + fname + ".mat")
        points = load_gt_points(gt_path)
        with Image.open(img_path) as im:
            w, h = im.size
        density = generate_density_map((h, w), points, adaptive=adaptive)
        np.save(out_path, density.astype(np.float32))


def load_density_map(density_dir, fname):
    return np.load(os.path.join(density_dir, fname + ".npy"))

In [ ]:
train_density_dir_by_part, test_density_dir_by_part = {}, {}
for p in active_parts:
    train_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_train")
    test_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_test")
    precompute_density_maps(train_image_paths_by_part[p], train_gt_dir_by_part[p],
                             train_density_dir_by_part[p], part=p)
    precompute_density_maps(test_image_paths_by_part[p], test_gt_dir_by_part[p],
                             test_density_dir_by_part[p], part=p)
    print(f"part_{p} density maps cached at:", train_density_dir_by_part[p],
          "and", test_density_dir_by_part[p])

## 5. Dataset & train/validation split (unchanged)

The train/validation split uses a fixed seed (42) that does **not** depend on the run seed, so
every variant is validated on exactly the same 45 images and tested on the same 182.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def pad_to_min_size(img_np, density_np, min_h, min_w):
    h, w = img_np.shape[:2]
    pad_h, pad_w = max(0, min_h - h), max(0, min_w - w)
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def pad_to_multiple(img_np, density_np, multiple):
    h, w = img_np.shape[:2]
    new_h = int(np.ceil(h / multiple) * multiple)
    new_w = int(np.ceil(w / multiple) * multiple)
    pad_h, pad_w = new_h - h, new_w - w
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def random_crop(img_np, density_np, patch_size):
    h, w = img_np.shape[:2]
    top = random.randint(0, h - patch_size)
    left = random.randint(0, w - patch_size)
    img_c = img_np[top:top + patch_size, left:left + patch_size, :]
    density_c = density_np[top:top + patch_size, left:left + patch_size]
    return img_c, density_c


def sum_pool(density, stride):
    # density: (B, 1, H, W) -> (B, 1, H/stride, W/stride), sum-preserving
    pooled = F.avg_pool2d(density, kernel_size=stride, stride=stride)
    return pooled * (stride * stride)


def random_rescale(img_np, density_np, scale_range):
    # Count-preserving rescale: bilinear-resize the density map, then renormalise it so it still
    # sums to the original head count.
    s = random.uniform(*scale_range)
    h, w = img_np.shape[:2]
    new_h, new_w = max(1, int(round(h * s))), max(1, int(round(w * s)))
    if (new_h, new_w) == (h, w):
        return img_np, density_np
    count = float(density_np.sum())
    img_np = cv2.resize(img_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    density_np = cv2.resize(density_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    new_sum = float(density_np.sum())
    if new_sum > 0:
        density_np = density_np * (count / new_sum)
    return img_np, density_np.astype(np.float32)


class ShanghaiTechDataset(Dataset):
    def __init__(self, image_paths, density_dir, patch_size, output_stride,
                 density_scale, train=True, crops_per_image=1, scale_range=None,
                 gray_prob=0.0):
        self.image_paths = image_paths
        self.density_dir = density_dir
        self.patch_size = patch_size
        self.output_stride = output_stride
        self.density_scale = density_scale
        self.train = train
        self.crops_per_image = crops_per_image if train else 1
        self.scale_range = scale_range
        self.gray_prob = gray_prob

    def __len__(self):
        return len(self.image_paths) * self.crops_per_image

    def __getitem__(self, idx):
        img_path = self.image_paths[idx % len(self.image_paths)]
        fname = os.path.splitext(os.path.basename(img_path))[0]
        img = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)
        density = load_density_map(self.density_dir, fname).astype(np.float32)

        if self.train:
            if self.scale_range is not None:
                img, density = random_rescale(img, density, self.scale_range)
            img, density = pad_to_min_size(img, density, self.patch_size, self.patch_size)
            img, density = random_crop(img, density, self.patch_size)
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])
                density = np.ascontiguousarray(density[:, ::-1])
            if random.random() < self.gray_prob:
                img = np.repeat(np.asarray(Image.fromarray(img).convert("L"))[..., None], 3, axis=2)
        else:
            # Only pad to a multiple of output_stride (< 8 px) so the sum-pooled GT loses no heads
            # at the border. Do NOT pad to a multiple of patch_size here: that reflect-padding
            # duplicated real people into a region whose GT density is zero, inflating the error.
            img, density = pad_to_multiple(img, density, self.output_stride)

        img_tensor = _to_tensor_norm(img)
        density_tensor = torch.from_numpy(density.copy()).unsqueeze(0).unsqueeze(0).float()
        density_tensor = density_tensor * self.density_scale
        density_ds = sum_pool(density_tensor, self.output_stride).squeeze(0)
        raw_count = torch.tensor(float(density.sum()), dtype=torch.float32)
        return img_tensor, density_ds, raw_count

In [ ]:
def stratified_train_val_split(image_paths, val_split, seed=42):
    idx = list(range(len(image_paths)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(len(idx) * val_split))
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return [image_paths[i] for i in train_idx], [image_paths[i] for i in val_idx]


train_datasets, val_datasets, test_datasets_by_part = [], [], {}
for p in active_parts:
    part_train_paths, part_val_paths = stratified_train_val_split(
        train_image_paths_by_part[p], cfg.val_split)
    train_datasets.append(ShanghaiTechDataset(
        part_train_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=True,
        crops_per_image=cfg.crops_per_image, scale_range=cfg.scale_aug,
        gray_prob=cfg.gray_aug_prob))
    val_datasets.append(ShanghaiTechDataset(
        part_val_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False))
    test_datasets_by_part[p] = ShanghaiTechDataset(
        test_image_paths_by_part[p], test_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False)
    print(f"part_{p}: {len(part_train_paths)} train (x{cfg.crops_per_image} crops/epoch) / {len(part_val_paths)} val "
          f"/ {len(test_datasets_by_part[p])} test")

# A single part just falls back to its own dataset; "both" concatenates across parts so the
# rest of the notebook (training loop, diagnostics, visualisation) needs no further changes --
# ConcatDataset supports the same __len__/__getitem__ interface as a plain Dataset.
train_dataset = train_datasets[0] if len(train_datasets) == 1 else ConcatDataset(train_datasets)
val_dataset = val_datasets[0] if len(val_datasets) == 1 else ConcatDataset(val_datasets)
test_dataset = (next(iter(test_datasets_by_part.values())) if len(test_datasets_by_part) == 1
                else ConcatDataset(list(test_datasets_by_part.values())))

print(f"\ntotal: train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True,
                          persistent_workers=cfg.num_workers > 0, drop_last=True)
# Validation/test images keep their native (padded) resolution, which varies per sample,
# so they are loaded one at a time and processed with tiled sliding-window inference.
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
# Kept separately too so Section 11 can report per-part test metrics alongside the combined one.
test_loaders_by_part = {
    p: DataLoader(ds, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    for p, ds in test_datasets_by_part.items()
}

In [ ]:
def make_train_loader(seed):
    # Re-created per run so each variant sees the same, seed-controlled shuffling order.
    g = torch.Generator()
    g.manual_seed(seed)

    def _worker_init(worker_id):
        s = (seed * 1000 + worker_id) % (2 ** 32)
        np.random.seed(s)
        random.seed(s)

    return DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=True,
                      persistent_workers=cfg.num_workers > 0, drop_last=True,
                      generator=g, worker_init_fn=_worker_init)


def dataset_image_name(dataset, idx):
    if isinstance(dataset, ConcatDataset):
        prev = 0
        for sub, cum in zip(dataset.datasets, dataset.cumulative_sizes):
            if idx < cum:
                return os.path.basename(sub.image_paths[idx - prev])
            prev = cum
    return os.path.basename(dataset.image_paths[idx])

## 6. Model
### 6.1 Building blocks (unchanged)

In [ ]:
class PositionalEncoding2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        channels = int(np.ceil(channels / 4) * 2)
        self.channels = channels
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2).float() / channels))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        b, c, h, w = x.shape
        pos_h = torch.arange(h, device=x.device, dtype=self.inv_freq.dtype)
        pos_w = torch.arange(w, device=x.device, dtype=self.inv_freq.dtype)
        sin_inp_h = torch.einsum("i,j->ij", pos_h, self.inv_freq)
        sin_inp_w = torch.einsum("i,j->ij", pos_w, self.inv_freq)
        emb_h = torch.cat([sin_inp_h.sin(), sin_inp_h.cos()], dim=-1)  # (H, channels)
        emb_w = torch.cat([sin_inp_w.sin(), sin_inp_w.cos()], dim=-1)  # (W, channels)
        emb = torch.zeros(2 * self.channels, h, w, device=x.device, dtype=x.dtype)
        emb[:self.channels] = emb_h.t().unsqueeze(2).expand(-1, -1, w)
        emb[self.channels:2 * self.channels] = emb_w.t().unsqueeze(1).expand(-1, h, -1)
        emb = emb[:c].unsqueeze(0)
        return x + emb


class DilatedPyramidPooling(nn.Module):
    """ASPP-style module: parallel dilated-conv branches at several rates plus a
    global-pooling branch, fused together -- covers both the 'dilated convolutions'
    and 'pyramid structure' multi-scale-context requirements in one block."""

    def __init__(self, in_channels, out_channels, dilations=(1, 2, 3, 4)):
        super().__init__()
        branch_ch = out_channels // (len(dilations) + 1)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            ) for d in dilations
        ])
        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, branch_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )
        total_ch = branch_ch * (len(dilations) + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        g = self.global_branch(x)
        g = F.interpolate(g, size=(h, w), mode="bilinear", align_corners=False)
        feats.append(g)
        return self.fuse(torch.cat(feats, dim=1))


class SpatialTransformer(nn.Module):
    """Learns a per-image affine warp so downstream layers see a perspective/scale
    -normalised feature map -- useful because crowd scenes are shot from many angles
    and distances.

    [PERF FIX] The warped features are added back as a *residual* (through a zero-initialised
    1x1 conv) instead of replacing the input. The predicted density map must stay pixel-aligned
    with the ground truth; when the warp replaced the features outright, any learned zoom/shift
    moved the predicted heads away from their GT locations (and `padding_mode="border"`
    smeared edge features across the vacated area), which hurt both MSE and the count. The
    residual form keeps an aligned identity path while still letting the STN contribute."""

    def __init__(self, in_channels):
        super().__init__()
        self.loc_net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(16 * 4 * 4, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 6),
        )
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float32))
        self.proj = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        xs = self.loc_net(x)
        xs = xs.reshape(xs.size(0), -1)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        # affine_grid/grid_sample are done in float32 for accurate sampling coordinates
        grid = F.affine_grid(theta.float(), x.size(), align_corners=False)
        warped = F.grid_sample(x.float(), grid, align_corners=False, padding_mode="zeros").to(x.dtype)
        return x + self.proj(warped)


class TransformerContext(nn.Module):
    """Global self-attention over the (coarse, stride-8) spatial grid so every location
    can attend to crowd context anywhere in the patch."""

    def __init__(self, channels, num_layers=2, num_heads=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.pos_enc = PositionalEncoding2D(channels)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = self.pos_enc(x).flatten(2).transpose(1, 2)  # (B, H*W, C)
        out = self.transformer(tokens)
        return out.transpose(1, 2).reshape(b, c, h, w)


class GraphOcclusionModule(nn.Module):
    """Graph-based occlusion handling: pools the feature map to a coarse grid of
    'region' nodes, builds a soft top-k similarity graph between them, and propagates
    messages so a heavily-occluded region can borrow evidence from visually-similar,
    less-occluded regions elsewhere in the scene."""

    def __init__(self, channels, grid_size=16, k=8):
        super().__init__()
        self.grid_size = grid_size
        self.k = k
        self.pool = nn.AdaptiveAvgPool2d(grid_size)
        self.node_transform = nn.Linear(channels, channels)
        self.message_transform = nn.Linear(channels, channels)
        self.update = nn.Sequential(nn.Linear(channels * 2, channels), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        b, c, h, w = x.shape
        g = self.grid_size
        nodes = self.pool(x)
        nodes_flat = nodes.flatten(2).transpose(1, 2)  # (B, N, C), N = g*g

        q = self.node_transform(nodes_flat)
        q_norm = F.normalize(q, dim=-1)
        sim = torch.bmm(q_norm, q_norm.transpose(1, 2))  # (B, N, N)

        topk = min(self.k, sim.size(-1))
        topk_val, topk_idx = sim.topk(topk, dim=-1)
        mask = torch.full_like(sim, float("-inf"))
        mask.scatter_(-1, topk_idx, topk_val)
        adj = F.softmax(mask, dim=-1)

        messages = self.message_transform(nodes_flat)
        agg = torch.bmm(adj, messages)
        updated = self.update(torch.cat([nodes_flat, agg], dim=-1))
        updated = updated.transpose(1, 2).reshape(b, c, g, g)

        updated_up = F.interpolate(updated, size=(h, w), mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([x, updated_up], dim=1))

### 6.2 Ablatable model

Same architecture as `HybridCNNTransformerCrowdCounter` in the main notebook, with a switch per
component. A removed module is **not constructed at all** (replaced by an identity), so the
parameter and FLOP counts in the results table reflect the real cost of each module.

With every switch on, this is exactly the main notebook's model — the `full` row should report
**11.07M parameters**, which is a quick sanity check.

In [ ]:
class AblatableCrowdCounter(nn.Module):
    def __init__(self, pretrained=True, freeze_stage1=True, transformer_layers=2,
                 transformer_heads=4, transformer_ff_dim=512, graph_grid_size=16,
                 graph_k=8, use_grad_checkpoint=False,
                 use_multiscale=True, use_dppm=True, use_stn=True,
                 use_transformer=True, use_graph=True):
        super().__init__()
        weights = tv_models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
        vgg_features = tv_models.vgg16(weights=weights).features

        self.stage3 = vgg_features[:16]   # -> stride 4,  256 channels (conv3_3 + relu)
        self.stage4 = vgg_features[16:23]  # -> stride 8,  512 channels (pool3 + conv4 block)

        if freeze_stage1:
            for p in vgg_features[:10].parameters():  # conv1 + conv2 blocks
                p.requires_grad_(False)

        self.use_multiscale = use_multiscale
        self.lateral4 = nn.Conv2d(512, 256, kernel_size=1)
        if use_multiscale:
            self.lateral3 = nn.Conv2d(256, 256, kernel_size=1)
            self.down3 = nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1)  # stride4 -> stride8
            fuse_in = 512
        else:
            # Single-scale: only conv4_3 features; same BN+ReLU projection so the head is unchanged.
            fuse_in = 256
        self.fuse_ms = nn.Sequential(
            nn.Conv2d(fuse_in, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.dppm = DilatedPyramidPooling(256, 256) if use_dppm else nn.Identity()
        self.stn = SpatialTransformer(256) if use_stn else nn.Identity()
        self.transformer = (TransformerContext(256, transformer_layers, transformer_heads,
                                               transformer_ff_dim)
                            if use_transformer else nn.Identity())
        self.graph = GraphOcclusionModule(256, graph_grid_size, graph_k) if use_graph else nn.Identity()

        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU(inplace=True),  # density values must be non-negative
        )

        self.use_grad_checkpoint = use_grad_checkpoint

    def _maybe_checkpoint(self, module, x):
        if self.use_grad_checkpoint and self.training and not isinstance(module, nn.Identity):
            return torch.utils.checkpoint.checkpoint(module, x, use_reentrant=False)
        return module(x)

    def forward(self, x):
        # Backbone under mixed precision; everything after it in float32 (as in the main notebook).
        f3 = self._maybe_checkpoint(self.stage3, x)
        f4 = self._maybe_checkpoint(self.stage4, f3)

        with torch.autocast(device_type=x.device.type, enabled=False):
            f3, f4 = f3.float(), f4.float()
            f4_p = self.lateral4(f4)
            if self.use_multiscale:
                f3_ds = self.down3(self.lateral3(f3))
                fused = self.fuse_ms(torch.cat([f3_ds, f4_p], dim=1))
            else:
                fused = self.fuse_ms(f4_p)

            x = self.dppm(fused)
            x = self.stn(x)
            x = self._maybe_checkpoint(self.transformer, x)
            x = self.graph(x)
            return self.decoder(x)


def get_model_for_state_dict(model):
    # torch.compile wraps the model; always save/load through the unwrapped module.
    return getattr(model, "_orig_mod", model)


_ALL_ON = dict(use_multiscale=True, use_dppm=True, use_stn=True, use_transformer=True, use_graph=True)
_ALL_OFF = {k: False for k in _ALL_ON}

VARIANTS = {
    # ---- leave-one-out ----
    "full":           dict(desc="Full model",                model={}),
    "no_multiscale":  dict(desc="w/o multi-scale fusion",    model=dict(use_multiscale=False)),
    "no_dppm":        dict(desc="w/o DPPM",                  model=dict(use_dppm=False)),
    "no_stn":         dict(desc="w/o STN",                   model=dict(use_stn=False)),
    "no_transformer": dict(desc="w/o Transformer",           model=dict(use_transformer=False)),
    "no_graph":       dict(desc="w/o graph occlusion",       model=dict(use_graph=False)),
    "baseline":       dict(desc="Baseline (VGG16 + decoder)", model=dict(_ALL_OFF)),
    # ---- cumulative (optional) ----
    "inc_ms":             dict(desc="Baseline + MS",
                               model=dict(_ALL_OFF, use_multiscale=True)),
    "inc_ms_dppm":        dict(desc="Baseline + MS + DPPM",
                               model=dict(_ALL_OFF, use_multiscale=True, use_dppm=True)),
    "inc_ms_dppm_stn":    dict(desc="Baseline + MS + DPPM + STN",
                               model=dict(_ALL_OFF, use_multiscale=True, use_dppm=True, use_stn=True)),
    "inc_ms_dppm_stn_tr": dict(desc="Baseline + MS + DPPM + STN + Transformer",
                               model=dict(_ALL_ON, use_graph=False)),
    # ---- loss terms (optional; full architecture) ----
    "loss_mse_only":  dict(desc="Loss: MSE only",            model={}, loss=dict(lambda_ssim=0.0, lambda_count=0.0)),
    "loss_no_ssim":   dict(desc="Loss: w/o SSIM term",       model={}, loss=dict(lambda_ssim=0.0)),
    "loss_no_count":  dict(desc="Loss: w/o count term",      model={}, loss=dict(lambda_count=0.0)),
}
unknown = [v for v in RUN_VARIANTS if v not in VARIANTS]
assert not unknown, f"Unknown variant(s) in RUN_VARIANTS: {unknown}"


def build_model(flags):
    return AblatableCrowdCounter(
        pretrained=cfg.pretrained_backbone,
        freeze_stage1=cfg.freeze_backbone_stage1,
        transformer_layers=cfg.transformer_layers,
        transformer_heads=cfg.transformer_heads,
        transformer_ff_dim=cfg.transformer_ff_dim,
        graph_grid_size=cfg.graph_grid_size,
        graph_k=cfg.graph_k,
        use_grad_checkpoint=cfg.use_grad_checkpoint,
        **flags,
    ).to(DEVICE)


@torch.no_grad()
def model_complexity(model):
    # Parameters, and GFLOPs for one patch_size x patch_size input (the unit of tiled inference).
    # FlopCounterMode counts conv/matmul/attention FLOPs; cheap element-wise ops (grid_sample,
    # top-k, softmax) are not included, which is standard practice for reported FLOPs.
    n_params = sum(p.numel() for p in model.parameters())
    gflops = float("nan")
    try:
        from torch.utils.flop_counter import FlopCounterMode
        was_training = model.training
        model.eval()
        x = torch.randn(1, 3, cfg.patch_size, cfg.patch_size, device=DEVICE)
        with FlopCounterMode(display=False) as fc:
            model(x)
        gflops = fc.get_total_flops() / 1e9
        model.train(was_training)
    except Exception as e:
        print("FLOP counting unavailable:", e)
    return n_params, gflops


print(f"{'variant':20s} {'params (M)':>10s} {'GFLOPs/patch':>13s}")
for name in RUN_VARIANTS:
    m = build_model(VARIANTS[name]["model"])
    n, g = model_complexity(m)
    print(f"{name:20s} {n/1e6:10.2f} {g:13.2f}")
    del m
torch.cuda.empty_cache()

## 7. Loss, tiled inference, train/validate loops (unchanged)

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, channel=1, sigma=1.5):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        gauss = torch.tensor(
            [math.exp(-((x - window_size // 2) ** 2) / float(2 * sigma ** 2)) for x in range(window_size)]
        )
        gauss = gauss / gauss.sum()
        _2d = gauss.unsqueeze(1) @ gauss.unsqueeze(0)
        window = _2d.float().unsqueeze(0).unsqueeze(0).expand(channel, 1, window_size, window_size)
        self.register_buffer("window", window.contiguous())

    def forward(self, pred, target):
        window = self.window.to(dtype=pred.dtype)
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        pad = self.window_size // 2
        mu1 = F.conv2d(pred, window, padding=pad, groups=self.channel)
        mu2 = F.conv2d(target, window, padding=pad, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(pred * pred, window, padding=pad, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target * target, window, padding=pad, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred * target, window, padding=pad, groups=self.channel) - mu1_mu2
        ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / \
                   ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2) + 1e-12)
        return 1 - ssim_map.mean()


class CrowdCountingLoss(nn.Module):
    def __init__(self, lambda_mse=1.0, lambda_ssim=0.1, lambda_count=0.02):
        super().__init__()
        self.lambda_mse = lambda_mse
        self.lambda_ssim = lambda_ssim
        self.lambda_count = lambda_count
        self.mse = nn.MSELoss()
        self.ssim = SSIMLoss()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        ssim_loss = self.ssim(pred, target)
        pred_count = pred.sum(dim=[1, 2, 3])
        target_count = target.sum(dim=[1, 2, 3])
        count_loss = F.l1_loss(pred_count, target_count)
        total = (self.lambda_mse * mse_loss
                 + self.lambda_ssim * ssim_loss
                 + self.lambda_count * count_loss)
        parts = {
            "mse": mse_loss.item(), "ssim": ssim_loss.item(),
            "count": count_loss.item(), "total": total.item(),
        }
        return total, parts


def compute_mae_mse(pred, target, density_scale):
    pred_count = pred.sum(dim=[1, 2, 3]) / density_scale
    target_count = target.sum(dim=[1, 2, 3]) / density_scale
    mae = (pred_count - target_count).abs()
    se = (pred_count - target_count) ** 2
    return mae, se


criterion = CrowdCountingLoss(cfg.lambda_mse, cfg.lambda_ssim, cfg.lambda_count).to(DEVICE)

In [ ]:
def _tile_starts(length, patch_size, step):
    if length <= patch_size:
        return [0]
    starts = list(range(0, length - patch_size + 1, step))
    if starts[-1] != length - patch_size:
        starts.append(length - patch_size)  # last tile flush with the image edge
    return starts


@torch.no_grad()
def _tiled_predict_single(model, img, patch_size, output_stride, device, amp, chunk_size,
                          overlap=0.5, bn_train_mode=False):
    # bn_train_mode is only used by the Section 10.2 diagnostic (forces BatchNorm to use each
    # chunk's own batch statistics); normal inference always runs in eval mode.
    model.train(bn_train_mode)
    img = img.to(device)
    _, _, H, W = img.shape  # H, W are multiples of output_stride

    # Images smaller than a tile are padded with zeros -- i.e. the ImageNet-mean colour after
    # normalisation, a neutral people-free region -- and the prediction is cropped back below.
    pad_h, pad_w = max(0, patch_size - H), max(0, patch_size - W)
    if pad_h or pad_w:
        img = F.pad(img, (0, pad_w, 0, pad_h), value=0.0)
    Hp, Wp = img.shape[-2:]

    step = int(patch_size * (1 - overlap)) // output_stride * output_stride
    step = max(output_stride, min(step, patch_size))
    positions = [(y, x) for y in _tile_starts(Hp, patch_size, step)
                 for x in _tile_starts(Wp, patch_size, step)]

    tile_out = patch_size // output_stride
    pred_sum = torch.zeros(1, 1, Hp // output_stride, Wp // output_stride, device=device)
    weight = torch.zeros_like(pred_sum)

    # Tiles are batched (chunk_size at a time) into single forward passes -- far better GPU
    # utilisation than one tile per call -- while capping peak memory regardless of image size.
    for start in range(0, len(positions), chunk_size):
        chunk_pos = positions[start:start + chunk_size]
        batch = torch.cat([img[:, :, y:y + patch_size, x:x + patch_size] for y, x in chunk_pos], dim=0)
        n = batch.size(0)
        if bn_train_mode:
            if n == 1:  # train-mode BatchNorm needs > 1 value per channel
                batch = batch.repeat(2, 1, 1, 1)
        elif n < chunk_size:
            # Pad the last, partial chunk to a fixed batch size: in eval mode the extra rows don't
            # affect the real ones, and a constant input shape stops torch.compile from
            # recompiling for every leftover chunk size.
            batch = torch.cat([batch, batch.new_zeros(chunk_size - n, *batch.shape[1:])], dim=0)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(amp and device.type == "cuda")):
            out = model(batch)
        out = out.float()[:n]
        for (y, x), p in zip(chunk_pos, out):
            oy, ox = y // output_stride, x // output_stride
            pred_sum[:, :, oy:oy + tile_out, ox:ox + tile_out] += p.unsqueeze(0)
            weight[:, :, oy:oy + tile_out, ox:ox + tile_out] += 1

    model.eval()
    pred = pred_sum / weight.clamp_min(1.0)
    return pred[:, :, :H // output_stride, :W // output_stride]


@torch.no_grad()
def tiled_predict(model, img, patch_size, output_stride, device, amp=True, tta=False, chunk_size=8,
                  overlap=None):
    overlap = cfg.infer_overlap if overlap is None else overlap
    pred = _tiled_predict_single(model, img, patch_size, output_stride, device, amp, chunk_size, overlap)
    if tta:
        # Average with the horizontal-flip prediction (flipped back to align) -- a cheap,
        # well-established accuracy boost at test/inference time, at 2x the compute cost.
        flipped = torch.flip(img, dims=[3])
        pred_flipped = _tiled_predict_single(model, flipped, patch_size, output_stride, device,
                                              amp, chunk_size, overlap)
        pred = (pred + torch.flip(pred_flipped, dims=[3])) / 2
    return pred

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, cfg, epoch):
    model.train()
    running = {"loss": 0.0, "mae": 0.0, "mse": 0.0, "ssim": 0.0, "count": 0.0}
    n_samples, n_skipped, n_consecutive_bad = 0, 0, 0
    optimizer.zero_grad()
    pbar = tqdm(loader, desc=f"Epoch {epoch} [train]", leave=False)
    for step, (imgs, densities, _) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(cfg.amp and device.type == "cuda")):
            preds = model(imgs)
        # The loss (squares/divisions inside SSIM) is computed in float32 regardless of the
        # autocast dtype used for the forward pass -- squaring already density_scale-scaled
        # values can overflow float16's ~65504 max, which otherwise silently turns into NaN.
        loss, parts = criterion(preds.float(), densities.float())

        if not torch.isfinite(loss):
            n_skipped += 1
            n_consecutive_bad += 1
            if n_skipped <= 3:
                # Say where the NaN/inf comes from instead of just that it happened.
                print(f"  [warn] non-finite loss at epoch {epoch} step {step} -- skipping batch "
                      f"(inputs finite={torch.isfinite(imgs).all().item()}, "
                      f"targets finite={torch.isfinite(densities).all().item()}, "
                      f"preds finite={torch.isfinite(preds).all().item()}, "
                      f"weights finite={all(torch.isfinite(p).all().item() for p in model.parameters())})")
            optimizer.zero_grad(set_to_none=True)
            if n_consecutive_bad >= 20:
                raise RuntimeError(
                    f"20 consecutive non-finite batches in epoch {epoch}: training is not recoverable "
                    "this way. If 'weights finite=False' above, reload a checkpoint (Section 10.1); "
                    "otherwise the forward pass itself overflows -- set cfg.amp=False to confirm.")
            continue
        n_consecutive_bad = 0

        loss_scaled = loss / cfg.grad_accum_steps
        scaler.scale(loss_scaled).backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            mae, _ = compute_mae_mse(preds.float(), densities.float(), cfg.density_scale)

        bs = imgs.size(0)
        running["loss"] += parts["total"] * bs
        running["mae"] += mae.sum().item()
        running["mse"] += parts["mse"] * bs
        running["ssim"] += parts["ssim"] * bs
        running["count"] += parts["count"] * bs
        n_samples += bs
        pbar.set_postfix(loss=f"{parts['total']:.4f}", mae=f"{mae.mean().item():.2f}",
                          lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    if n_skipped:
        print(f"  Epoch {epoch}: skipped {n_skipped} non-finite batch(es).")
    torch.cuda.empty_cache()
    n_samples = max(n_samples, 1)
    return {k: v / n_samples for k, v in running.items()}


@torch.no_grad()
def validate(model, loader, criterion, device, cfg, desc="Validate", tta=False):
    model.eval()
    running = {"loss": 0.0, "mae": 0.0, "se": 0.0, "mse": 0.0, "ssim": 0.0, "count": 0.0}
    n = 0
    for imgs, densities, _ in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)
        preds = tiled_predict(model, imgs, cfg.patch_size, cfg.output_stride, device,
                               amp=cfg.amp, tta=tta, chunk_size=cfg.batch_size)
        loss, parts = criterion(preds, densities)
        mae, se = compute_mae_mse(preds, densities, cfg.density_scale)

        bs = imgs.size(0)
        running["loss"] += parts["total"] * bs
        running["mae"] += mae.sum().item()
        running["se"] += se.sum().item()
        running["mse"] += parts["mse"] * bs
        running["ssim"] += parts["ssim"] * bs
        running["count"] += parts["count"] * bs
        n += bs

    torch.cuda.empty_cache()
    return {
        "loss": running["loss"] / n, "mae": running["mae"] / n,
        "rmse": math.sqrt(running["se"] / n), "mse": running["mse"] / n,
        "ssim": running["ssim"] / n, "count": running["count"] / n,
    }

## 8. Run the ablation

Each run (`<variant>_seed<seed>`) gets its own folder under `RUN_ROOT` containing:

- `best.pth` — weights with the lowest validation MAE
- `last.pth` — full resume state (weights, optimiser, scheduler, history, RNG), saved every epoch
- `history.json` — per-epoch curves
- `test_per_image.csv` — ground-truth and predicted count for every test image (used for the
  significance tests)
- `result.json` — final metrics; its presence marks the run as finished

The training loop is the main notebook's `run_training` (warmup, smoothed-val-MAE
`ReduceLROnPlateau`, differential backbone LR, best-by-val-MAE checkpointing), extended with
resume support.

In [ ]:
HISTORY_KEYS = ["train_loss", "val_loss", "train_mae", "val_mae", "lr",
                "train_mse", "train_ssim", "train_count", "val_mse", "val_ssim", "val_count"]


def build_optimizer(model):
    m = get_model_for_state_dict(model)
    backbone_params = [p for p in list(m.stage3.parameters()) + list(m.stage4.parameters())
                       if p.requires_grad]
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in m.parameters() if p.requires_grad and id(p) not in backbone_ids]
    head_lr, backbone_lr = cfg.base_lr, cfg.base_lr * cfg.backbone_lr_mult
    return torch.optim.AdamW([
        {"params": head_params, "lr": head_lr, "group_base_lr": head_lr},
        {"params": backbone_params, "lr": backbone_lr, "group_base_lr": backbone_lr},
    ], weight_decay=cfg.weight_decay)


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience, min_lr=cfg.lr_min
    )


def set_warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    factor = min(1.0, epoch / max(1, warmup_epochs))
    for pg in optimizer.param_groups:
        pg["lr"] = pg.get("group_base_lr", base_lr) * factor


def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history["train_mae"], label="train")
    axes[1].plot(history["val_mae"], label="val")
    axes[1].set_title("MAE (count error)"); axes[1].set_xlabel("epoch"); axes[1].legend()
    axes[2].plot(history["lr"])
    axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch"); axes[2].set_yscale("log")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def _rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def _set_rng_state(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu())
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def _atomic_save(obj, path):
    # Write to a temp file first so a disconnect mid-write never leaves a corrupt checkpoint.
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


@torch.no_grad()
def evaluate_per_image(model, dataset, tta, desc):
    model.eval()
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)
    rows = []
    for i, (img, dens, _) in enumerate(tqdm(loader, desc=desc, leave=False)):
        pred = tiled_predict(model, img, cfg.patch_size, cfg.output_stride, DEVICE,
                             amp=cfg.amp, tta=tta, chunk_size=cfg.batch_size)
        gt_c = dens.sum().item() / cfg.density_scale
        pr_c = pred.sum().item() / cfg.density_scale
        rows.append({"image": dataset_image_name(dataset, i), "gt": gt_c, "pred": pr_c,
                     "abs_err": abs(pr_c - gt_c)})
    errs = np.array([r["pred"] - r["gt"] for r in rows])
    return rows, float(np.abs(errs).mean()), float(np.sqrt((errs ** 2).mean()))


@torch.no_grad()
def time_inference(model, dataset, n_images=20, warmup=3):
    # Mean wall-clock time per full test image (overlapping tiled inference, no TTA).
    if DEVICE.type != "cuda":
        return float("nan")
    model.eval()
    imgs = [dataset[i][0].unsqueeze(0) for i in range(min(n_images, len(dataset)))]
    for img in imgs[:warmup]:
        tiled_predict(model, img, cfg.patch_size, cfg.output_stride, DEVICE, amp=cfg.amp,
                      tta=False, chunk_size=cfg.batch_size)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for img in imgs:
        tiled_predict(model, img, cfg.patch_size, cfg.output_stride, DEVICE, amp=cfg.amp,
                      tta=False, chunk_size=cfg.batch_size)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / len(imgs) * 1000


def run_variant(name, seed):
    spec = VARIANTS[name]
    run_name = f"{name}_seed{seed}"
    run_dir = os.path.join(RUN_ROOT, run_name)
    result_path = os.path.join(run_dir, "result.json")
    if os.path.exists(result_path):
        print(f"[skip] {run_name} already finished.")
        with open(result_path) as f:
            return json.load(f)
    os.makedirs(run_dir, exist_ok=True)
    best_path = os.path.join(run_dir, "best.pth")
    last_path = os.path.join(run_dir, "last.pth")

    set_seed(seed)
    if hasattr(torch, "_dynamo"):
        torch._dynamo.reset()  # fresh compile cache per variant (different graph each time)
    raw_model = build_model(spec["model"])
    n_params, gflops = model_complexity(raw_model)
    model = raw_model
    if cfg.use_torch_compile and DEVICE.type == "cuda":
        try:
            model = torch.compile(raw_model)
        except Exception as e:
            print("torch.compile unavailable, continuing uncompiled:", e)

    loss_kw = dict(lambda_mse=cfg.lambda_mse, lambda_ssim=cfg.lambda_ssim,
                   lambda_count=cfg.lambda_count)
    loss_kw.update(spec.get("loss", {}))
    run_criterion = CrowdCountingLoss(**loss_kw).to(DEVICE)

    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and DEVICE.type == "cuda"
                                                   and AMP_DTYPE == torch.float16))
    history = {k: [] for k in HISTORY_KEYS}
    best_val_mae, best_epoch, start_epoch = float("inf"), 0, 1

    if os.path.exists(last_path):
        st = torch.load(last_path, map_location=DEVICE, weights_only=False)
        get_model_for_state_dict(model).load_state_dict(st["model_state"])
        optimizer.load_state_dict(st["optimizer_state"])
        scheduler.load_state_dict(st["scheduler_state"])
        history = st["history"]
        best_val_mae, best_epoch = st["best_val_mae"], st["best_epoch"]
        start_epoch = st["epoch"] + 1
        _set_rng_state(st["rng"])
        print(f"[resume] {run_name} from epoch {start_epoch} (best val MAE so far "
              f"{best_val_mae:.2f} @ epoch {best_epoch})")

    print(f"=== {run_name}: {spec['desc']} | {n_params/1e6:.2f}M params | {gflops:.2f} GFLOPs/patch "
          f"| loss {loss_kw}")
    # A resumed run gets a different (but still deterministic) shuffling seed.
    train_loader_run = make_train_loader(seed * 10007 + start_epoch)
    run_start = time.time()

    for epoch in range(start_epoch, cfg.epochs + 1):
        in_warmup = epoch <= cfg.warmup_epochs
        if in_warmup:
            set_warmup_lr(optimizer, epoch, cfg.warmup_epochs, cfg.base_lr)

        train_stats = train_one_epoch(model, train_loader_run, optimizer, run_criterion,
                                      scaler, DEVICE, cfg, epoch)
        val_stats = validate(model, val_loader, run_criterion, DEVICE, cfg)

        if not in_warmup:
            recent = history["val_mae"][-(cfg.val_mae_smoothing - 1):] if cfg.val_mae_smoothing > 1 else []
            scheduler.step(float(np.mean(recent + [val_stats["mae"]])))
        current_lr = optimizer.param_groups[0]["lr"]

        for k, v in [("train_loss", train_stats["loss"]), ("val_loss", val_stats["loss"]),
                     ("train_mae", train_stats["mae"]), ("val_mae", val_stats["mae"]),
                     ("lr", current_lr), ("train_mse", train_stats["mse"]),
                     ("train_ssim", train_stats["ssim"]), ("train_count", train_stats["count"]),
                     ("val_mse", val_stats["mse"]), ("val_ssim", val_stats["ssim"]),
                     ("val_count", val_stats["count"])]:
            history[k].append(v)

        improved = val_stats["mae"] < best_val_mae
        if improved:
            best_val_mae, best_epoch = val_stats["mae"], epoch
            _atomic_save({"epoch": epoch, "variant": name, "seed": seed,
                          "model_flags": spec["model"], "val_mae": val_stats["mae"],
                          "model_state": get_model_for_state_dict(model).state_dict()}, best_path)
        _atomic_save({"epoch": epoch, "model_state": get_model_for_state_dict(model).state_dict(),
                      "optimizer_state": optimizer.state_dict(),
                      "scheduler_state": scheduler.state_dict(), "history": history,
                      "best_val_mae": best_val_mae, "best_epoch": best_epoch,
                      "rng": _rng_state()}, last_path)

        done = epoch - start_epoch + 1
        sec_per_epoch = (time.time() - run_start) / done
        eta_h = sec_per_epoch * (cfg.epochs - epoch) / 3600
        if epoch % PLOT_EVERY == 0 or epoch == cfg.epochs:
            clear_output(wait=True)
            plot_history(history, f"{run_name} — {spec['desc']}")
        flag = " ** best **" if improved else ""
        print(f"[{run_name}] epoch {epoch}/{cfg.epochs} | train_mae={train_stats['mae']:.2f} "
              f"val_mae={val_stats['mae']:.2f} val_rmse={val_stats['rmse']:.2f} | lr={current_lr:.1e} "
              f"| {sec_per_epoch/60:.1f} min/epoch, ETA {eta_h:.1f} h{flag}")

    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f)

    # ---- test the best checkpoint ----
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    get_model_for_state_dict(model).load_state_dict(best["model_state"])
    rows_tta, mae_tta, rmse_tta = evaluate_per_image(model, test_dataset, tta=True,
                                                     desc=f"{run_name} test (TTA)")
    _, mae_plain, rmse_plain = evaluate_per_image(model, test_dataset, tta=False,
                                                  desc=f"{run_name} test")
    ms_per_img = time_inference(model, test_dataset)

    with open(os.path.join(run_dir, "test_per_image.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["image", "gt", "pred", "abs_err"])
        w.writeheader()
        w.writerows(rows_tta)

    result = {
        "variant": name, "seed": seed, "desc": spec["desc"],
        "model_flags": spec["model"], "loss_weights": loss_kw,
        "params_M": n_params / 1e6, "trainable_params_M":
            sum(p.numel() for p in raw_model.parameters() if p.requires_grad) / 1e6,
        "gflops_per_patch": gflops, "ms_per_image": ms_per_img,
        "epochs": cfg.epochs, "best_epoch": best_epoch, "best_val_mae": best_val_mae,
        "test_mae": mae_tta, "test_rmse": rmse_tta,
        "test_mae_no_tta": mae_plain, "test_rmse_no_tta": rmse_plain,
        "part": cfg.part, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    }
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    if not KEEP_LAST_CHECKPOINT and os.path.exists(last_path):
        os.remove(last_path)

    print(f"\n[done] {run_name}: test MAE={mae_tta:.2f} RMSE={rmse_tta:.2f} (TTA) | "
          f"no-TTA MAE={mae_plain:.2f} RMSE={rmse_plain:.2f} | best epoch {best_epoch} | "
          f"{ms_per_img:.0f} ms/image")
    del model, raw_model, optimizer
    torch.cuda.empty_cache()
    return result

In [ ]:
# Runs every (variant, seed) in the queue. Safe to re-run after a disconnect: finished runs are
# skipped and an interrupted run resumes from its last saved epoch.
all_results = []
for seed in SEEDS:
    for name in RUN_VARIANTS:
        all_results.append(run_variant(name, seed))

print("\nAll requested runs finished.")
for r in all_results:
    print(f"  {r['variant']:20s} seed={r['seed']}  test MAE={r['test_mae']:.2f}  RMSE={r['test_rmse']:.2f}")

## 9. Results for the manuscript

These cells only read the `result.json` / `test_per_image.csv` files on Drive, so they can be
run in a fresh session without re-training anything.

### 9.1 Ablation table (CSV + LaTeX)

In [ ]:
_results = []
for path in sorted(glob.glob(os.path.join(RUN_ROOT, "*", "result.json"))):
    with open(path) as f:
        _results.append(json.load(f))
assert _results, f"No finished runs found under {RUN_ROOT}"
res_df = pd.DataFrame(_results)

metrics = ["test_mae", "test_rmse", "test_mae_no_tta", "test_rmse_no_tta", "best_val_mae",
           "best_epoch", "params_M", "gflops_per_patch", "ms_per_image"]
agg = res_df.groupby("variant").agg(
    desc=("desc", "first"), n_seeds=("seed", "nunique"),
    **{f"{m}_mean": (m, "mean") for m in metrics},
    **{f"{m}_std": (m, "std") for m in ["test_mae", "test_rmse"]},
)
order = [v for v in VARIANTS if v in agg.index]
agg = agg.loc[order]
if "full" in agg.index:
    agg["delta_mae"] = agg["test_mae_mean"] - agg.loc["full", "test_mae_mean"]
    agg["delta_rmse"] = agg["test_rmse_mean"] - agg.loc["full", "test_rmse_mean"]

show_cols = ["desc", "n_seeds", "params_M_mean", "gflops_per_patch_mean", "ms_per_image_mean",
             "best_epoch_mean", "best_val_mae_mean", "test_mae_mean", "test_mae_std",
             "test_rmse_mean", "test_rmse_std", "delta_mae", "delta_rmse",
             "test_mae_no_tta_mean", "test_rmse_no_tta_mean"]
table = agg[[c for c in show_cols if c in agg.columns]].round(2)
display(table)

table.to_csv(os.path.join(RUN_ROOT, "ablation_results.csv"))
res_df.to_csv(os.path.join(RUN_ROOT, "ablation_results_per_seed.csv"), index=False)


def _fmt(mean, std, multi):
    return f"{mean:.2f} $\\pm$ {std:.2f}" if multi and not np.isnan(std) else f"{mean:.2f}"


multi_seed = int(agg["n_seeds"].max()) > 1
lines = [
    r"\begin{table}[t]", r"\centering",
    rf"\caption{{Ablation study on ShanghaiTech Part {cfg.part} (test set, {cfg.epochs} epochs"
    + (rf", mean $\pm$ std over {int(agg['n_seeds'].max())} seeds" if multi_seed else "")
    + r"). Lower is better.}",
    r"\label{tab:ablation}",
    r"\begin{tabular}{lcccc}", r"\hline",
    r"Variant & Params (M) & GFLOPs & MAE & RMSE \\", r"\hline",
]
for v, row in agg.iterrows():
    name = row["desc"] + (" (ours)" if v == "full" else "")
    lines.append(f"{name} & {row['params_M_mean']:.2f} & {row['gflops_per_patch_mean']:.1f} & "
                 f"{_fmt(row['test_mae_mean'], row['test_mae_std'], multi_seed)} & "
                 f"{_fmt(row['test_rmse_mean'], row['test_rmse_std'], multi_seed)} \\\\")
lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
latex = "\n".join(lines)
with open(os.path.join(RUN_ROOT, "ablation_table.tex"), "w") as f:
    f.write(latex)
print(latex)
print("\nSaved ablation_results.csv, ablation_results_per_seed.csv and ablation_table.tex to", RUN_ROOT)

### 9.2 Ablation bar chart

In [ ]:
plot_df = agg.sort_values("test_mae_mean")
ACCENT, NEUTRAL, INK, MUTED = "#2a6fdb", "#a3a9b3", "#1f2328", "#59636e"

fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(plot_df) + 1.2))
y = np.arange(len(plot_df))
colors = [ACCENT if v == "full" else NEUTRAL for v in plot_df.index]
xerr = plot_df["test_mae_std"].fillna(0).values if multi_seed else None
ax.barh(y, plot_df["test_mae_mean"], height=0.6, color=colors, xerr=xerr,
        error_kw=dict(ecolor=MUTED, lw=1, capsize=3))
ax.set_yticks(y)
ax.set_yticklabels([d + (" (ours)" if v == "full" else "") for v, d in
                    zip(plot_df.index, plot_df["desc"])], color=INK)
ax.invert_yaxis()
xmax = float(plot_df["test_mae_mean"].max())
for yi, val in zip(y, plot_df["test_mae_mean"]):
    ax.text(val + xmax * 0.01, yi, f"{val:.1f}", va="center", fontsize=9, color=INK)
if "full" in plot_df.index:
    ax.axvline(plot_df.loc["full", "test_mae_mean"], color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Test MAE (lower is better)", color=INK)
ax.set_xlim(0, xmax * 1.12)
ax.grid(axis="x", color="#e5e7eb", lw=0.8)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color("#c9ced6")
ax.tick_params(colors=MUTED)
ax.set_title(f"Ablation on ShanghaiTech Part {cfg.part}", color=INK, loc="left")
plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RUN_ROOT, f"ablation_mae.{ext}"), dpi=300, bbox_inches="tight")
plt.show()

### 9.3 Paired significance test

With 182 test images, a difference of a few MAE points can be noise. The Wilcoxon signed-rank
test compares each variant's per-image absolute error with the full model's **on the same
images** (pooled over seeds when there are several). Report the p-values alongside the table;
p < 0.05 means the component's contribution is statistically significant.

In [ ]:
from scipy.stats import wilcoxon

sig_rows = []
for v in [v for v in agg.index if v != "full"]:
    full_err, var_err = [], []
    for seed in sorted(res_df.loc[res_df.variant == v, "seed"].unique()):
        f_path = os.path.join(RUN_ROOT, f"full_seed{seed}", "test_per_image.csv")
        v_path = os.path.join(RUN_ROOT, f"{v}_seed{seed}", "test_per_image.csv")
        if not (os.path.exists(f_path) and os.path.exists(v_path)):
            continue
        merged = pd.read_csv(f_path).merge(pd.read_csv(v_path), on="image", suffixes=("_full", "_var"))
        full_err += merged["abs_err_full"].tolist()
        var_err += merged["abs_err_var"].tolist()
    if not full_err:
        continue
    stat, p = wilcoxon(var_err, full_err)
    sig_rows.append({"variant": v, "desc": VARIANTS[v]["desc"], "n_pairs": len(full_err),
                     "mean_abs_err_full": np.mean(full_err), "mean_abs_err_variant": np.mean(var_err),
                     "full_better_on_%_images": 100 * np.mean(np.array(full_err) < np.array(var_err)),
                     "wilcoxon_p": p})
sig_df = pd.DataFrame(sig_rows).round(4)
display(sig_df)
sig_df.to_csv(os.path.join(RUN_ROOT, "ablation_significance.csv"), index=False)

### 9.4 Qualitative figure (full model)

Test images at four crowd densities (lowest, median, 90th percentile and highest ground-truth
count) with ground-truth and predicted density maps from the `full` model of this study.

In [ ]:
QUAL_SEED = SEEDS[0]
qual_ckpt = os.path.join(RUN_ROOT, f"full_seed{QUAL_SEED}", "best.pth")
qual_model = build_model(VARIANTS["full"]["model"])
qual_model.load_state_dict(torch.load(qual_ckpt, map_location=DEVICE, weights_only=False)["model_state"])
qual_model.eval()

per_img = pd.read_csv(os.path.join(RUN_ROOT, f"full_seed{QUAL_SEED}", "test_per_image.csv"))
name_to_idx = {dataset_image_name(test_dataset, i): i for i in range(len(test_dataset))}
per_img = per_img.sort_values("gt").reset_index(drop=True)
picks = sorted({0, len(per_img) // 2, int(0.9 * (len(per_img) - 1)), len(per_img) - 1})

fig, axes = plt.subplots(len(picks), 3, figsize=(13, 3.6 * len(picks)))
for r, k in enumerate(picks):
    idx = name_to_idx[per_img.loc[k, "image"]]
    img, dens, _ = test_dataset[idx]
    with torch.no_grad():
        pred = tiled_predict(qual_model, img.unsqueeze(0), cfg.patch_size, cfg.output_stride,
                             DEVICE, amp=cfg.amp, tta=cfg.use_tta, chunk_size=cfg.batch_size)
    vis = (img.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)).clip(0, 1)
    gt_map = dens.squeeze().numpy() / cfg.density_scale
    pr_map = pred.squeeze().cpu().numpy() / cfg.density_scale
    vmax = max(gt_map.max(), pr_map.max())
    axes[r, 0].imshow(vis); axes[r, 0].set_title(per_img.loc[k, "image"])
    axes[r, 1].imshow(gt_map, cmap="jet", vmin=0, vmax=vmax); axes[r, 1].set_title(f"Ground truth: {gt_map.sum():.0f}")
    axes[r, 2].imshow(pr_map, cmap="jet", vmin=0, vmax=vmax); axes[r, 2].set_title(f"Predicted: {pr_map.sum():.0f}")
    for a in axes[r]:
        a.axis("off")
plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RUN_ROOT, f"qualitative_full_model.{ext}"), dpi=200, bbox_inches="tight")
plt.show()
del qual_model
torch.cuda.empty_cache()

## 10. Reporting checklist

- State the protocol: *"All variants were trained for N epochs with identical hyper-parameters,
  data split and seed(s); the checkpoint with the lowest validation MAE was evaluated on the 182
  test images with horizontal-flip test-time augmentation."*
- Report **ΔMAE / ΔRMSE** relative to the full model, plus parameters and GFLOPs so the
  accuracy/cost trade-off of each module is visible.
- Include the Wilcoxon p-values (Section 9.3). If a module's removal changes MAE by less than
  the seed-to-seed standard deviation, say so honestly rather than claiming a contribution.
- Everything needed for the paper is on Drive in `crowd_counting_ablation/`:
  `ablation_results.csv`, `ablation_table.tex`, `ablation_significance.csv`,
  `ablation_mae.pdf`, `qualitative_full_model.pdf`, and each run's `history.json`
  (for training-curve figures).